<a href="https://colab.research.google.com/github/minperez/Actividad8/blob/main/Actividad8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Descripción del Proyecto

Este reporte aborda el flujo de trabajo de Machine Learning utilizando el clásico dataset Auto MPG (con información sobre el consumo de combustible y especificaciones de vehículos) https://archive.ics.uci.edu/dataset/9/auto+mpg

El reporte incluyo el algoritmo no lineal Random Forest utilizando el conjunto de datos Auto MPG (con información sobre el consumo y especificaciones de vehículos).

In [10]:
%pip install pandas numpy scikit-learn mlflow evidently requests
import time
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ==========================================
# 1. DATA PREPROCESSING & LOADING
# ==========================================
def load_auto_mpg_data():
    """
    Downloads and prepares the Auto MPG dataset directly from UCI repository.
    """
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
    columns = [
        "mpg", "cylinders", "displacement", "horsepower",
        "weight", "acceleration", "model_year", "origin", "car_name"
    ]

    # Read raw whitespace-delimited data
    df = pd.read_csv(url, names=columns, delim_whitespace=True, na_values="?")

    # Handle missing values (horsepower has 6 missing values)
    df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())

    # Feature engineering: car_name is text identifier, drop for training
    df = df.drop(columns=["car_name"])

    X = df.drop(columns=["mpg"])
    y = df["mpg"]

    return train_test_split(X, y, test_size=0.2, random_state=42), X.columns.tolist()

# ==========================================
# 2. MODEL TRAINING & MLFLOW REGISTRATION
# ==========================================
def train_and_register_model():
    mlflow.set_experiment("Auto_MPG_Proactive_Monitoring")

    (X_train, X_test, y_train, y_test), feature_names = load_auto_mpg_data()

    n_estimators = 100
    max_depth = 10
    random_state = 42

    with mlflow.start_run() as run:
        # Measure training duration
        start_time = time.time()
        rf = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=random_state
        )
        rf.fit(X_train, y_train)
        training_time_sec = round(time.time() - start_time, 4)

        # Predictions & Metrics Calculation
        y_pred = rf.predict(X_test)

        mse = float(mean_squared_error(y_test, y_pred))
        rmse = float(np.sqrt(mse))
        mae = float(mean_absolute_error(y_test, y_pred))
        r2 = float(r2_score(y_test, y_pred))

        # Log Hyperparameters
        mlflow.log_params({
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "random_state": random_state,
            "test_size": 0.2
        })

        # Log Metrics
        mlflow.log_metrics({
            "R2": r2,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "training_time_seconds": training_time_sec
        })

        # Register Scikit-learn Model Artifact
        mlflow.sklearn.log_model(
            sk_model=rf,
            artifact_path="model",
            registered_model_name="AutoMPG_RandomForest_Regressor"
        )

        print("=== Model Training & MLflow Registration Complete ===")
        print(f"Run ID: {run.info.run_id}")
        print(f"Metrics -> R2: {r2:.4f} | MAE: {mae:.4f} | RMSE: {rmse:.4f} | Time: {training_time_sec}s")

        return rf, X_train, X_test, y_train, y_test, run.info.run_id

# ==========================================
# 3. PROACTIVE MONITORING ENGINE
# ==========================================
class ProactiveModelMonitor:
    """
    Monitors data drift, feature distribution changes, and metric degradations.
    Triggers automated alerts when thresholds are breached.
    """
    def __init__(self, reference_data: pd.DataFrame, min_r2_threshold=0.80, max_mae_threshold=2.5):
        self.reference_data = reference_data
        self.min_r2 = min_r2_threshold
        self.max_mae = max_mae_threshold

    def trigger_alert(self, alert_type: str, message: str):
        """Mock alert notification dispatcher (Slack/PagerDuty/Email integration endpoint)."""
        print(f"\n🚨 [PROACTIVE ALERT - {alert_type.upper()}] 🚨")
        print(f"Details: {message}")
        print("Action Suggested: Initiate automated pipeline re-training or inspect input data pipelines.\n")

    def check_performance_degradation(self, current_r2: float, current_mae: float, run_id: str):
        """Monitors key production regression metrics against SLA limits."""
        mlflow.log_metric("production_R2", current_r2)
        mlflow.log_metric("production_MAE", current_mae)

        if current_r2 < self.min_r2:
            self.trigger_alert(
                "Performance Degradation",
                f"Production R² scored {current_r2:.4f}, falling below threshold ({self.min_r2})."
            )

        if current_mae > self.max_mae:
            self.trigger_alert(
                "Error Threshold Exceeded",
                f"Production MAE reached {current_mae:.4f}, exceeding max limit ({self.max_mae})."
            )

    def check_feature_drift(self, production_data: pd.DataFrame, alpha=0.05):
        """
        Uses Kolmogorov-Smirnov test to detect distribution shift in numerical features
        between training baseline and incoming production batches.
        """
        from scipy.stats import ks_2samp

        drift_detected = False
        for col in self.reference_data.columns:
            stat, p_value = ks_2samp(self.reference_data[col], production_data[col])
            if p_value < alpha:
                drift_detected = True
                self.trigger_alert(
                    "Data Drift Detected",
                    f"Feature '{col}' shifted significantly (KS Stat: {stat:.4f}, p-value: {p_value:.4e} < {alpha})."
                )
        if not drift_detected:
            print("✅ Data Drift Check Passed: Production distributions match baseline within tolerance.")

# ==========================================
# 4. EXECUTION & SIMULATED MONITORING RUN
# ==========================================
if __name__ == "__main__":
    # Step 1: Train & Register
    model, X_train, X_test, y_train, y_test, run_id = train_and_register_model()

    # Step 2: Initialize Proactive Monitor
    monitor = ProactiveModelMonitor(
        reference_data=X_train,
        min_r2_threshold=0.80,
        max_mae_threshold=2.50
    )

    # Step 3: Simulate Normal Production Data Processing
    print("\n--- Running Monitoring on Normal Production Data ---")
    y_prod_pred = model.predict(X_test)
    prod_r2 = r2_score(y_test, y_prod_pred)
    prod_mae = mean_absolute_error(y_test, y_prod_pred)

    monitor.check_performance_degradation(prod_r2, prod_mae, run_id)
    monitor.check_feature_drift(X_test)

    # Step 4: Simulate Drifting Production Batch (e.g., Vehicle Weights spike drastically)
    print("\n--- Simulating Production Feature Drift & Performance Degradation ---")
    X_test_drifted = X_test.copy()
    X_test_drifted["weight"] = X_test_drifted["weight"] * 1.85  # Heavy vehicle shift

    y_drifted_pred = model.predict(X_test_drifted)
    drifted_r2 = r2_score(y_test, y_drifted_pred)
    drifted_mae = mean_absolute_error(y_test, y_drifted_pred)

    monitor.check_feature_drift(X_test_drifted)
    monitor.check_performance_degradation(drifted_r2, drifted_mae, run_id)

2026/07/20 02:42:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/20 02:42:45 INFO mlflow.store.db.utils: Updating database tables
2026/07/20 02:42:48 INFO mlflow.tracking.fluent: Experiment with name 'Auto_MPG_Proactive_Monitoring' does not exist. Creating a new experiment.
/tmp/ipykernel_774/1651728518.py:25: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(url, names=columns, delim_whitespace=True, na_values="?")
2026/07/20 02:42:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


=== Model Training & MLflow Registration Complete ===
Run ID: de8d88a293c04424821beaf8f04a5b09
Metrics -> R2: 0.9140 | MAE: 1.5977 | RMSE: 2.1505 | Time: 0.3919s

--- Running Monitoring on Normal Production Data ---
✅ Data Drift Check Passed: Production distributions match baseline within tolerance.

--- Simulating Production Feature Drift & Performance Degradation ---

🚨 [PROACTIVE ALERT - DATA DRIFT DETECTED] 🚨
Details: Feature 'weight' shifted significantly (KS Stat: 0.7109, p-value: 4.8990e-32 < 0.05).
Action Suggested: Initiate automated pipeline re-training or inspect input data pipelines.


🚨 [PROACTIVE ALERT - PERFORMANCE DEGRADATION] 🚨
Details: Production R² scored 0.7618, falling below threshold (0.8).
Action Suggested: Initiate automated pipeline re-training or inspect input data pipelines.


🚨 [PROACTIVE ALERT - ERROR THRESHOLD EXCEEDED] 🚨
Details: Production MAE reached 2.7749, exceeding max limit (2.5).
Action Suggested: Initiate automated pipeline re-training or inspect 

Successfully registered model 'AutoMPG_RandomForest_Regressor'.
Created version '1' of model 'AutoMPG_RandomForest_Regressor'.
